# Training in Google Colab – Learning-based Autofocus for Holography

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliTUHH/master_thesis_autofocus/blob/main/notebooks/colab_training.ipynb)

Dieses Notebook führt die komplette Pipeline des Repos auf einer Colab-GPU aus:
Repo klonen → Python-3.11-Umgebung → Daten erzeugen → Training → TensorBoard → Evaluation → Ergebnisse nach Google Drive.

**Vor dem Start: GPU-Laufzeit wählen.** Menü *Laufzeit → Laufzeittyp ändern → Hardwarebeschleuniger: T4 GPU* (Free-Tier) und *Speichern*. Ohne GPU läuft alles auch, nur langsamer.

**Warum eine eigene Python-Umgebung?** Colab läuft (Stand 2026-10, Runtime 2026.07) mit Python 3.12. `holowizard` (DESY) verlangt Python ≥ 3.11 und < 3.12. Deshalb legen wir mit `uv` ein separates 3.11-venv im Projektordner an und starten alle Repo-Befehle mit `.venv/bin/python`. Die Notebook-Zellen selbst laufen weiterhin im Colab-Kernel (3.12) – das ist unkritisch, weil der Kernel nur Shell-Befehle absetzt.

**Konventionen** (siehe `docs/04_werkzeuge_und_workflow.md`):
- `RUN_NAME = "YYYY-MM-DD_<kurzname>_s<seed>"`
- Nach dem Lauf: `results.json` + Figuren sichern, Commit-SHA und Datensatz-Hash ins Logbuch `docs/notizen/experimente.md` eintragen.
- Checkpoints (`best.pt`) und Datensätze nie ins Git-Repo, nur nach Drive.


In [ ]:
# Zelle 2 – GPU prüfen (bei fehlender GPU erscheint eine Fehlermeldung von nvidia-smi; das Notebook läuft trotzdem weiter)
!nvidia-smi || echo "Keine GPU zugewiesen – Laufzeit -> Laufzeittyp ändern -> T4 GPU wählen (optional)."


In [ ]:
# Zelle 3 – Google Drive mounten (optional, aber empfohlen: Ergebnisse überleben so das Session-Ende)
USE_DRIVE = True  # auf False setzen, wenn kein Drive genutzt werden soll

DRIVE_DIR = None
if USE_DRIVE:
    try:
        from google.colab import drive  # nur in Colab verfügbar
        drive.mount("/content/drive")
        DRIVE_DIR = "/content/drive/MyDrive/masterarbeit_autofocus"
        import os
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print("Drive gemountet, Ablageordner:", DRIVE_DIR)
    except ImportError:
        print("Nicht in Colab – Drive wird übersprungen.")
else:
    print("Drive deaktiviert (USE_DRIVE = False).")


In [ ]:
# Zelle 4 – Repository klonen (oder aktualisieren) und hineinwechseln
import os

REPO_URL = "https://github.com/AliTUHH/master_thesis_autofocus.git"
REPO_DIR = "/content/master_thesis_autofocus"
BRANCH = "main"  # für Experimente auf einem Feature-Branch hier ändern

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print("Repo existiert bereits – hole Änderungen.")
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
!git log -1 --format="Commit: %H%nDatum:  %ci%nTitel:  %s"


In [ ]:
# Zelle 5 – Python-3.11-Umgebung mit uv anlegen und Abhängigkeiten installieren
# Colab setzt UV_*-Umgebungsvariablen, die uv-Aufrufe stören (siehe googlecolab/colabtools#5237).
# Wir räumen sie weg und zwingen uv, Python 3.11 bei Bedarf herunterzuladen.
import os, shutil, subprocess, sys

for var in ["UV_CONSTRAINT", "UV_BUILD_CONSTRAINT", "UV_PRERELEASE", "UV_SYSTEM_PYTHON", "UV_PYTHON_DOWNLOADS"]:
    os.environ.pop(var, None)
os.environ["UV_PYTHON_DOWNLOADS"] = "automatic"

if shutil.which("uv") is None:
    print("uv nicht gefunden – installiere per pip …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

!uv --version
!uv venv --python 3.11 .venv
VENV_PY = os.path.abspath(".venv/bin/python")
assert os.path.exists(VENV_PY), "venv wurde nicht angelegt – Ausgabe von 'uv venv' oben prüfen."
!{VENV_PY} --version

# Abhängigkeiten: bevorzugt requirements.txt des Repos. Torch kommt vom PyPI-Standardindex
# (holowizard pinnt torch==2.10; das PyPI-Wheel bringt CUDA 12.x mit und nutzt die Colab-GPU).
if os.path.exists("requirements.txt"):
    !uv pip install --python {VENV_PY} -r requirements.txt
else:
    print("requirements.txt fehlt – Fallback-Zelle 5b ausführen.")


In [ ]:
# Zelle 5b – FALLBACK: nur ausführen, wenn Zelle 5 keine requirements.txt gefunden hat oder scheiterte
import os
VENV_PY = os.path.abspath(".venv/bin/python")
if not os.path.exists(VENV_PY):
    raise SystemExit("Kein .venv/bin/python – zuerst Zelle 5 ausführen.")
!uv pip install --python {VENV_PY} holowizard pytest pyyaml matplotlib tensorboard tqdm h5py


In [ ]:
# Zelle 5c – Umgebung prüfen: Python-Version, torch + GPU, holowizard
import os
VENV_PY = os.path.abspath(".venv/bin/python")
if not os.path.exists(VENV_PY):
    raise SystemExit("Kein .venv/bin/python – zuerst Zelle 5 ausführen.")
!{VENV_PY} -c "import sys, torch; print('python', sys.version.split()[0]); print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"
!{VENV_PY} -c "import holowizard; print('holowizard', getattr(holowizard, '__version__', 'ok'))"


In [ ]:
# Zelle 6 – Datenerzeugung (klein zum Testen; groß optional)
import os, hashlib, json
VENV_PY = os.path.abspath(".venv/bin/python")
if not os.path.exists(VENV_PY):
    raise SystemExit("Kein .venv/bin/python – zuerst Zelle 5 ausführen.")

DATA_CONFIG = "configs/data_small.yaml"     # GPU-Variante: "configs/data_p05.yaml"
DATA_NAME = os.path.splitext(os.path.basename(DATA_CONFIG))[0].replace("data_", "")  # = Feld `name` in der YAML
DATA_DIR = f"data/processed/{DATA_NAME}"    # entspricht dem Default <output_dir>/<name> aus der YAML
OVERWRITE = False                           # True: vorhandene train/val/test.hdf5 neu erzeugen (--overwrite)

if not os.path.exists(DATA_CONFIG):
    raise SystemExit(f"{DATA_CONFIG} fehlt im Repo – verfügbare Configs: {os.listdir('configs') if os.path.isdir('configs') else 'kein configs/-Ordner'}")

# generate_data bricht ohne --overwrite ab, wenn eine Split-Datei schon existiert -> vorhandenen Datensatz wiederverwenden
if os.path.exists(os.path.join(DATA_DIR, "meta.json")) and not OVERWRITE:
    print(f"Datensatz {DATA_DIR} existiert bereits (meta.json gefunden) – Erzeugung übersprungen; OVERWRITE = True erzwingt sie.")
else:
    OVERWRITE_FLAG = "--overwrite" if OVERWRITE else ""
    !{VENV_PY} -m src.data.generate_data --config {DATA_CONFIG} --out {DATA_DIR} {OVERWRITE_FLAG}

# Datensatz-Fingerabdruck für das Logbuch (meta.json-Hash)
meta_path = os.path.join(DATA_DIR, "meta.json")
if os.path.exists(meta_path):
    META_HASH = hashlib.sha256(open(meta_path, "rb").read()).hexdigest()[:16]
    print("meta.json sha256 (gekürzt):", META_HASH)
    print(json.dumps(json.load(open(meta_path)), indent=2)[:1500])
else:
    print("Hinweis: keine meta.json gefunden unter", DATA_DIR)
!ls -la {DATA_DIR}   # erwartet: train.hdf5, val.hdf5, test.hdf5, train/val/test.json (Forge-Config), meta.json


In [ ]:
# Zelle 7 – Training
import os, datetime, subprocess
VENV_PY = os.path.abspath(".venv/bin/python")
if not os.path.exists(VENV_PY):
    raise SystemExit("Kein .venv/bin/python – zuerst Zelle 5 ausführen.")

TRAIN_CONFIG = "configs/base.yaml"
SEED = 0
KURZNAME = "cnn-baseline"            # kurz, kleingeschrieben, Bindestriche
RUN_NAME = f"{datetime.date.today().isoformat()}_{KURZNAME}_s{SEED}"
COMMIT = subprocess.run(["git", "rev-parse", "--short=12", "HEAD"], capture_output=True, text=True).stdout.strip()
print("RUN_NAME:", RUN_NAME, "| Commit:", COMMIT, "| Daten:", DATA_DIR)

!{VENV_PY} -m src.train --config {TRAIN_CONFIG} --data-dir {DATA_DIR} --run-name {RUN_NAME}

RUN_DIR = f"runs/{RUN_NAME}"   # paths.runs_dir aus configs/base.yaml ist "runs"
# erwartet: best.pt, config.yaml, history.json, results.json (Test-Metriken des besten Checkpoints),
#           loss_curves.png, test_scatter_target.png, test_scatter_z01_mm.png, test_error_vs_z01.png,
#           test_examples.png, test_predictions.npz, tb/ (TensorBoard)
!ls -la {RUN_DIR} || echo "Run-Ordner nicht gefunden – Trainingsausgabe oben prüfen."


In [ ]:
# Zelle 8 – TensorBoard inline (liest die Event-Dateien unter runs/)
%load_ext tensorboard
%tensorboard --logdir runs


In [ ]:
# Zelle 9 – Evaluation auf dem Testsplit
# src.evaluate schreibt eval_results.json, predictions.npz und die Plots (scatter_target, scatter_z01_mm,
# error_vs_z01, examples) nach <Checkpoint-Ordner>/eval_<Dateistamm>/, hier also runs/<run>/eval_test/.
import os, json
VENV_PY = os.path.abspath(".venv/bin/python")
CKPT = f"{RUN_DIR}/best.pt"
TEST_H5 = f"{DATA_DIR}/test.hdf5"
EVAL_DIR = os.path.join(RUN_DIR, "eval_" + os.path.splitext(os.path.basename(TEST_H5))[0])
for p in (VENV_PY, CKPT, TEST_H5):
    if not os.path.exists(p):
        raise SystemExit(f"Fehlt: {p} – vorherige Zellen prüfen.")

!{VENV_PY} -m src.evaluate --checkpoint {CKPT} --data {TEST_H5} --out {EVAL_DIR}

# Kennzahlen fürs Logbuch: MAE z01 [mm] und mittlerer relativer Fr-Fehler [%] auf dem Testsplit
MAE_MM = REL_FR = None
for name, path in (("results.json", os.path.join(RUN_DIR, "results.json")), ("eval_results.json", os.path.join(EVAL_DIR, "eval_results.json"))):
    if os.path.exists(path):
        data = json.load(open(path))
        phys = (data.get("test_metrics") or data.get("metrics") or {}).get("physical", {})
        if "z01_mae_mm" in phys:
            MAE_MM, REL_FR = phys["z01_mae_mm"], phys["fr_rel_err_mean_pct"]
        print(f"--- {path} ---")
        print(json.dumps(data, indent=2)[:2000])
print(f"\nTestsplit: MAE z01 = {MAE_MM} mm, rel. Fr-Fehler = {REL_FR} %")


In [ ]:
# Zelle 10 – Ergebnisse sichern: results.json + Figuren (+ optional Checkpoint) nach Drive
import os, shutil, glob
if DRIVE_DIR is None:
    print("Kein Drive gemountet (Zelle 3) – Ergebnisse liegen nur unter", RUN_DIR, "und gehen mit der Session verloren.")
else:
    dest = os.path.join(DRIVE_DIR, "runs", RUN_NAME)
    os.makedirs(dest, exist_ok=True)
    kept = []
    # Run-Ordner: results.json, history.json, config.yaml, loss_curves.png, test_*.png, test_predictions.npz;
    # eval_<split>/: eval_results.json, predictions.npz, Plots
    for pattern in ("*.json", "*.yaml", "*.yml", "*.txt", "*.png", "*.pdf", "*.npz", "eval_*/*.json", "eval_*/*.png", "eval_*/*.npz"):
        for src in glob.glob(os.path.join(RUN_DIR, pattern)):
            if os.path.isfile(src):
                os.makedirs(os.path.dirname(os.path.join(dest, os.path.relpath(src, RUN_DIR))), exist_ok=True)
                shutil.copy2(src, os.path.join(dest, os.path.relpath(src, RUN_DIR)))
                kept.append(os.path.relpath(src, RUN_DIR))
    SAVE_CHECKPOINT = True   # best.pt ist groß, aber für Resume/Weiterarbeit nützlich -> Drive, NIE ins Git
    if SAVE_CHECKPOINT and os.path.exists(os.path.join(RUN_DIR, "best.pt")):
        shutil.copy2(os.path.join(RUN_DIR, "best.pt"), os.path.join(dest, "best.pt"))
        kept.append("best.pt")
    # TensorBoard-Logs (runs/<run>/tb/) ebenfalls sichern (klein)
    for d in ("tb",):
        if os.path.isdir(os.path.join(RUN_DIR, d)):
            shutil.copytree(os.path.join(RUN_DIR, d), os.path.join(dest, d), dirs_exist_ok=True)
            kept.append(d + "/")
    print("Gesichert nach", dest)
    print("\n".join(sorted(kept)))

print("\nIns Repo gehören nur results.json, eval_test/eval_results.json + Figuren (runs/ ist gitignored -> git add -f), z. B.:")
print(f"  python tools/export_figures.py --run {RUN_DIR} --chapter experimente --names loss_curves test_scatter_z01_mm test_error_vs_z01")
print("Logbuch-Zeile für docs/notizen/experimente.md:")
_fmt = lambda v: "?" if v is None else f"{v:.2f}"
print(f"| {RUN_NAME} | {__import__('datetime').date.today().isoformat()} | {COMMIT} | {DATA_NAME} / {globals().get('META_HASH', '?')} | {TRAIN_CONFIG} | {_fmt(globals().get('MAE_MM'))} | {_fmt(globals().get('REL_FR'))} | ... | ... |")


## Zelle 11 – Session-Limits, Resume und Hygiene

**Limits (Stand 2026-10-01, Colab-FAQ):** Free-Tier-Sessions laufen höchstens 12 h und enden früher bei Inaktivität oder Ressourcenknappheit; Google veröffentlicht weder die Idle-Grenze noch eine GPU-Garantie (meist T4). Colab Pro (9,99 $/Monat, 100 Compute Units) bringt bessere GPUs und mehr Verfügbarkeit; nur Pro+ erlaubt Hintergrundausführung bis 24 h. Kaggle Notebooks sind die Alternative mit 30 GPU-h/Woche und 12-h-Sessions (P100 oder 2×T4).

**Konsequenzen für das Training**
- Plane Läufe so, dass sie in **< 8 h** fertig sind; größere Experimente in mehrere Läufe mit unterschiedlichen Seeds/Configs aufteilen.
- Sichere jeden Lauf direkt nach dem Ende mit Zelle 10 nach Drive. Alternativ den Run-Ordner gleich auf Drive schreiben: `paths.runs_dir` in einer Kopie von `configs/base.yaml` auf einen absoluten Drive-Pfad setzen (z. B. `/content/drive/MyDrive/masterarbeit_autofocus/runs`) und diese Config per `--config` übergeben; `src.train` hat keine `--out`-Option.
- `src.train` kennt (Stand jetzt) **kein Resume** (`python -m src.train --help`: `--config`, `--data-dir`, `--run-name`, `--epochs`, `--device`). Nach einem Abbruch fehlen `results.json` und die Test-Auswertung; `best.pt` wird aber nach jeder besseren Validierungsepoche geschrieben und lässt sich mit Zelle 9 (`src.evaluate`) auswerten. Lange Läufe lieber mit `--epochs` kürzen und als mehrere protokollierte Läufe planen.
- Browser-Tab offen lassen; die Session gilt sonst als inaktiv.

**Hygiene**
- Jeder Lauf: `RUN_NAME`, Commit-SHA, Datensatz (`meta.json`-Hash), Config-Änderung und Metriken ins Logbuch `docs/notizen/experimente.md`.
- Große Datensätze (`data_p05.yaml`) besser auf dem DESY-Maxwell-Cluster oder TUHH-Ressourcen über die Betreuer rechnen – Colab ist für Smoke-Tests und mittlere Experimente gedacht.
- Dieses Notebook ohne Ausgaben committen (*Bearbeiten → Alle Ausgaben löschen*), damit das Diff lesbar bleibt.
